[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/SCP/blob/main/1_setup.ipynb)

# SCP Step 1 - Setup Tune Directory

Prepare a tune directory for the rest of the pipeline. Step 1 can download or use an existing cell source, compile mechanisms, scaffold configs, and validate the result.

Workflow:
- **1.1 Select Tune Directory**: choose the cell/tune path and basic metadata.
- **1.2 Setup Cell Source**: download an ADB model bundle or use existing local model files.
- **1.3 Compile Mechanisms**: compile/load NEURON `modfiles/`.
- **1.4 Setup Base Configs**: create or update `cell_config.json`, `geometry.json`, and `sim_config.json`.
- **1.5 Setup Optional Synapse Configs**: scaffold `syn_config.json` and disabled synapse-group templates when needed.
- **1.6 Validate and Review**: check files, mechanisms, cell loading, and optional synapse/input configs.
- **1.7 Quick Path Check**: confirm the standard Step 1 files exist.

Use this notebook when creating a new tune or refreshing setup files before tuning/simulation. Detailed guide: `docs/guides/step_1_setup.md`.

In [ ]:
# Environment setup: works locally or in Google Colab
%load_ext autoreload
%autoreload 2

import json
import os
import subprocess
import sys
from pathlib import Path

# User-editable only when running in a fresh Colab or unusual local layout.
SCP_REPO_URL = os.environ.get("SCP_REPO_URL", "https://github.com/cyneuro/SCP.git")
SCP_REPO_BRANCH = os.environ.get("SCP_REPO_BRANCH", "") or None
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
AUTO_CLONE_REPO = os.environ.get("SCP_AUTO_CLONE", "1" if IN_COLAB else "0") not in {"0", "false", "False"}
INSTALL_DEPS = None  # None = install automatically in Colab, do not install locally.
SCP_REPO_DIR = Path(os.environ.get("SCP_REPO_DIR", "/content/SCP" if IN_COLAB else str(Path.cwd() / "SCP")))


def _looks_like_scp_repo(path: Path) -> bool:
    return (path / "modules").is_dir() and (path / "run_pipeline.py").is_file()


# Minimal pre-import bootstrap. Fresh Colab cannot import SCP helpers until the repo exists.
repo_root = None
env_root = os.environ.get("SCP_ROOT")
if env_root and _looks_like_scp_repo(Path(env_root).expanduser()):
    repo_root = Path(env_root).expanduser().resolve()
else:
    start = Path.cwd().resolve()
    candidates = list((start, *start.parents))
    for base in (start, start.parent):
        try:
            candidates.extend(child for child in base.iterdir() if child.is_dir())
        except Exception:
            pass
    for candidate in candidates:
        if _looks_like_scp_repo(candidate):
            repo_root = candidate.resolve()
            break

if repo_root is None:
    if not AUTO_CLONE_REPO:
        raise FileNotFoundError("Could not find SCP. Set SCP_ROOT or enable SCP_AUTO_CLONE=1.")
    clone_url = SCP_REPO_URL
    token = os.environ.get("SCP_GIT_TOKEN") or os.environ.get("SCP_GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN")
    if token and clone_url.startswith("https://") and "@" not in clone_url:
        clone_url = clone_url.replace("https://", f"https://{token}@", 1)
    cmd = ["git", "clone", "--depth", "1"]
    if SCP_REPO_BRANCH:
        cmd += ["--branch", SCP_REPO_BRANCH]
    cmd += [clone_url, str(SCP_REPO_DIR)]
    subprocess.check_call(cmd)
    repo_root = SCP_REPO_DIR.resolve()

os.environ["SCP_ROOT"] = str(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from modules.notebooks.bootstrap import finish_step1_notebook_setup

setup = finish_step1_notebook_setup(repo_root, install_deps=INSTALL_DEPS)
repo_root = setup["repo_root"]
IN_COLAB = setup["in_colab"]


## 1.1 Select Tune Directory

Choose the target cell/tune folder and basic metadata used when scaffolding configs.

Quick guide:
- `cell_name`: cell folder under `cells/`, for example `PV` or `SST`.
- `tune_name`: tune folder under `cells/<cell>/tunes/`.
- `tune_dir_override`: optional direct path; use this for nonstandard locations.
- `soma_diam_multiplier`: `None` uses SCP's known-cell default; set a number to force a tuning multiplier.
- `cell_color`: `None` uses the known-cell default; set a Matplotlib color string for plots/metadata.

In [ ]:
# Target tune location
cell_name = "SST"               # e.g., PV, SST, PN
tune_name = "adb_all"           # folder under cells/<cell>/tunes/
tunes_parent = "tunes"
tune_dir_override = None        # optional absolute/relative path; overrides cell/tune settings

# Cell metadata scaffold (set None to use known-cell defaults)
soma_diam_multiplier = None    # None uses known-cell default; or set an explicit multiplier
cell_color = None               # None uses known-cell default; or set e.g. "royalblue", "mediumorchid"

from modules.setup.step1_prepare import (
    guess_cell_color,
    guess_soma_multiplier,
)

if tune_dir_override:
    tune_dir = Path(tune_dir_override).expanduser().resolve()
else:
    tune_dir = (repo_root / "cells" / cell_name / tunes_parent / tune_name).resolve()

if soma_diam_multiplier is None:
    soma_diam_multiplier = guess_soma_multiplier(cell_name)
if cell_color is None:
    cell_color = guess_cell_color(cell_name)

print("Tune dir:", tune_dir)
print("Cell:", cell_name, "| Tune:", tune_name)
print("Soma diam multiplier:", soma_diam_multiplier, "| Color:", cell_color)


## 1.2 Setup Cell Source

Choose where the model files come from. ADB download is the primary built-in source, but existing local model files can also be used.

Quick guide:
- `source_type = "adb"`: download/cache an Allen Database bundle and write source metadata.
- `source_type = "existing"`: skip download and use files already staged in the tune directory.
- `specimen_id`: Allen specimen ID; `None` uses known-cell defaults when available.
- `model_type`: Allen model family such as `perisomatic` or `all active`.
- `LIST_AVAILABLE_ADB_MODELS`: prints matching Allen models without changing files.
- `DO_DOWNLOAD_ADB_BUNDLE`: set `False` when the bundle is already present.
- `FORCE_DOWNLOAD`: redownload even if files already exist.
- `CACHE_STIMULUS`: also cache the Allen stimulus file when available.

In [ ]:
# Source adapter
source_type = "adb"             # "adb" or "existing"

# ADB-specific settings
specimen_id = 485466109         # SST: 485466109, PV: 484635029; set None to use known-cell default
model_type = "all active"       # e.g., perisomatic, all active
LIST_AVAILABLE_ADB_MODELS = False
DO_DOWNLOAD_ADB_BUNDLE = True
FORCE_DOWNLOAD = False
CACHE_STIMULUS = False

from modules.setup.adb import list_ADB_models
from modules.setup.step1_prepare import guess_specimen_from_cell, prepare_cell_source

if source_type == "adb" and specimen_id is None:
    specimen_id = guess_specimen_from_cell(cell_name)
if source_type == "adb" and specimen_id is None:
    raise ValueError("specimen_id is required for ADB setup; set it explicitly.")

print("Source type:", source_type)
print("Specimen:", specimen_id, "| Model type:", model_type)

if source_type == "adb" and LIST_AVAILABLE_ADB_MODELS:
    list_ADB_models(specimen_id, filter_type=model_type)

cell_source_summary = prepare_cell_source(
    tune_dir=tune_dir,
    source_type=source_type,
    specimen_id=specimen_id,
    model_type=model_type,
    do_download=DO_DOWNLOAD_ADB_BUNDLE,
    force_download=FORCE_DOWNLOAD,
    cache_stimulus=CACHE_STIMULUS,
)
print(json.dumps(cell_source_summary, indent=2))


## 1.3 Compile Mechanisms

Compile `modfiles/` with `nrnivmodl` and optionally load the compiled mechanism library into the current kernel.

Quick guide:
- `DO_COMPILE_MODFILES`: compile mechanisms if needed.
- `RECOMPILE_MODFILES`: force a clean rebuild; use after editing `.mod` files.
- `LOAD_COMPILED_DLL`: load compiled mechanisms immediately for validation/loading.
- `SORT_GENOME_ENTRIES_BY_SECTION`: normalize some ADB fit JSON ordering issues.
- `COERCE_GENOME_VALUES_TO_NUMERIC`: fix numeric values stored as strings in some all-active ADB bundles.

In [ ]:
DO_COMPILE_MODFILES = True
RECOMPILE_MODFILES = False
LOAD_COMPILED_DLL = True
SORT_GENOME_ENTRIES_BY_SECTION = False
COERCE_GENOME_VALUES_TO_NUMERIC = False  # useful for some all-active ADB bundles

from modules.setup.step1_prepare import prepare_mechanisms

mechanisms_summary = prepare_mechanisms(
    tune_dir=tune_dir,
    do_compile_modfiles=DO_COMPILE_MODFILES,
    recompile_modfiles=RECOMPILE_MODFILES,
    load_compiled_dll=LOAD_COMPILED_DLL,
    coerce_genome_values_to_numeric=COERCE_GENOME_VALUES_TO_NUMERIC,
    sort_genome_entries_by_section=SORT_GENOME_ENTRIES_BY_SECTION,
)
print(json.dumps(mechanisms_summary, indent=2))


## 1.4 Setup Base Configs

Create or update first-level configs: `cell_config.json`, `geometry.json`, and `sim_config.json`.

Quick guide:
- `DO_SETUP_BASE_CONFIGS`: turn base config scaffolding on/off.
- `CONFIG_MODE = "fill"`: create missing files/fields without overwriting existing values.
- `CONFIG_MODE = "overwrite"`: regenerate files from defaults; use carefully.
- `CONFIG_MODE = "skip"`: leave existing config files untouched.
- `SYNC_CELL_METADATA`: update basic cell metadata such as name, tune, specimen ID, model type, color, and soma multiplier.

In [ ]:
DO_SETUP_BASE_CONFIGS = True
CONFIG_MODE = "fill"            # "fill" | "overwrite" | "skip"
SYNC_CELL_METADATA = True

from modules.setup.step1_prepare import prepare_base_configs

if DO_SETUP_BASE_CONFIGS:
    base_config_summary = prepare_base_configs(
        tune_dir=tune_dir,
        cell_name=cell_name,
        tune_name=tune_name,
        specimen_id=specimen_id,
        model_type=model_type,
        soma_diam_multiplier=soma_diam_multiplier,
        color=cell_color,
        config_mode=CONFIG_MODE,
        sync_cell_metadata=SYNC_CELL_METADATA,
    )
else:
    base_config_summary = {"status": "skipped"}

print(json.dumps(base_config_summary, indent=2))


## 1.5 Setup Optional Synapse Configs

Create or update `syn_config.json` and selected disabled template files under `syn_groups/`. Disable this for cell-only/current-injection workflows.

Quick guide:
- `DO_SETUP_SYNAPSE_CONFIGS`: create synapse config scaffolds when `True`.
- `SYN_CONFIG_MODE`: usually matches `CONFIG_MODE`; use `fill` to avoid overwriting edited groups.
- `SYN_GROUP_TEMPLATES`: choose which disabled example group templates to generate.
- `SYN_TEMPLATE_WEIGHT_STYLE = "distributed"`: generate `wt_mean`/`wt_std` example fields.
- `SYN_TEMPLATE_WEIGHT_STYLE = "fixed"`: generate a fixed `initW` example field.

Generated synapse templates are starting points. Add mechanism-specific synapse parameters manually after choosing the actual mod file and input mode.

In [ ]:
DO_SETUP_SYNAPSE_CONFIGS = True
SYN_CONFIG_MODE = CONFIG_MODE

# Generate one disabled example synapse group using explicit input_blocks.
# The template shows baseline -> stimulus curve -> baseline in one group.
SYN_GROUP_TEMPLATES = {
    "input_blocks": True,
}

# Generated templates use one generic weight style; add mechanism-specific params manually.
SYN_TEMPLATE_WEIGHT_STYLE = "distributed"  # "distributed" | "fixed"

from modules.setup.step1_prepare import prepare_synapse_configs

synapse_template_kinds = [
    name for name, enabled in SYN_GROUP_TEMPLATES.items() if enabled
]

if DO_SETUP_SYNAPSE_CONFIGS:
    synapse_config_summary = prepare_synapse_configs(
        tune_dir=tune_dir,
        config_mode=SYN_CONFIG_MODE,
        template_kinds=synapse_template_kinds,
        weight_style=SYN_TEMPLATE_WEIGHT_STYLE,
    )
else:
    synapse_config_summary = {"status": "skipped"}

print(json.dumps(synapse_config_summary, indent=2))


## 1.6 Validate and Review

Run lightweight checks for files, compiled mechanisms, cell loading, and optional synapse/input configs.

Quick guide:
- `DO_VALIDATE`: master toggle for validation.
- `VALIDATE_MODFILES`: check compiled mechanisms are present.
- `VALIDATE_LOAD_CELL`: instantiate the selected NEURON cell.
- `VALIDATE_SYNAPSE_CONFIGS`: check enabled synapse group config structure.
- `VALIDATE_INPUT_CONFIGS`: check input-block timing/source config structure.

Leave these enabled for normal setup. Disable individual checks only when intentionally working on incomplete pieces.

In [ ]:
DO_VALIDATE = True
VALIDATE_MODFILES = DO_COMPILE_MODFILES
VALIDATE_LOAD_CELL = True
VALIDATE_SYNAPSE_CONFIGS = DO_SETUP_SYNAPSE_CONFIGS
VALIDATE_INPUT_CONFIGS = DO_SETUP_SYNAPSE_CONFIGS

from modules.setup.step1_prepare import validate_setup

if DO_VALIDATE:
    validation_summary = validate_setup(
        tune_dir=tune_dir,
        cell_name=cell_name,
        soma_diam_multiplier=soma_diam_multiplier,
        validate_modfiles=VALIDATE_MODFILES,
        validate_load_cell=VALIDATE_LOAD_CELL,
        validate_inputs_cfg=VALIDATE_INPUT_CONFIGS,
        validate_synapses=VALIDATE_SYNAPSE_CONFIGS,
    )
else:
    validation_summary = {"status": "skipped"}

print(json.dumps(validation_summary, indent=2))


## 1.7 Quick Path Check

Print the standard files produced by Step 1. Synapse config files are optional.

This is a read-only review cell. If anything is `MISSING`, rerun the relevant setup cell above or inspect the selected `tune_dir`.

In [ ]:
check_paths = [
    tune_dir / "manifest.json",
    tune_dir / "modfiles",
    tune_dir / "cell_configs" / "cell_config.json",
    tune_dir / "cell_configs" / "sim_config.json",
    tune_dir / "cell_configs" / "geometry.json",
]
if DO_SETUP_SYNAPSE_CONFIGS:
    template_files = {
        "input_blocks": "input_blocks_template.json",
    }
    check_paths.append(tune_dir / "cell_configs" / "syn_config.json")
    check_paths.extend(
        tune_dir / "cell_configs" / "syn_groups" / template_files[name]
        for name in synapse_template_kinds
    )

for path in check_paths:
    print(f"{'OK' if path.exists() else 'MISSING'}  {path}")
